# 07_negocio — Escenario de valor para EPS e IPS

Este notebook construye el componente de negocio del proyecto: traduce la matriz analítica de RNA-Seq y los resultados del modelado en un escenario operativo comprensible para EPS e IPS.

El objetivo es estimar, de forma reproducible, cómo una solución analítica basada en expresión génica y aprendizaje automático podría apoyar la clasificación de tipos de cáncer, priorizar rutas diagnósticas, reducir tiempos estimados de atención y disminuir reprocesos operativos.

Esta versión incorpora una simulación diferenciada por EPS, de modo que las visualizaciones no queden homogéneas: algunas EPS concentran mayor oportunidad de mejora, otras muestran menor costo final y otras presentan mayor ahorro promedio estimado.

> **Nota metodológica:** las variables de EPS, IPS, costos y tiempos se construyen como un escenario simulado para comunicación analítica y evaluación de valor. No representan datos reales de aseguramiento, facturación o atención clínica individual.


## 1. Configuración del entorno

Se reutiliza la configuración del proyecto en Databricks: Unity Catalog, Volumes, tablas Delta refined y carpetas de salida para visualizaciones y app exports.


In [0]:
%run ./00_configuracion_databricks

In [0]:

from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

# Rutas de salida sobre el Volume configurado en 00_configuracion_databricks
VIZ_PATH = REFINED_VISUALIZATIONS_PATH / "negocio_eps_ips"
APP_EXPORTS_PATH = REFINED_APP_EXPORTS_PATH / "negocio_eps_ips"

# Entradas principales en capa refined
MATRIZ_MODELADO_TABLE = "refined_ml_matriz_modelado_desde_feature_selection"
PREDICCIONES_TEST_TABLE = "refined_predicciones_test_mejor_modelo"

# Salidas de negocio en capa refined
CONTEXTO_NEGOCIO_TABLE = "refined_contexto_negocio_eps_ips_cohorte"
RESUMEN_EJECUTIVO_TABLE = "refined_resumen_ejecutivo_eps_ips_cohorte"
RESUMEN_EPS_TABLE = "refined_resumen_por_eps_cohorte"
RESUMEN_CANCER_TABLE = "refined_resumen_por_tipo_cancer_cohorte"

for path in [VIZ_PATH, APP_EXPORTS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Ruta refined:", REFINED_PATH)
print("Ruta visualizaciones negocio:", VIZ_PATH)
print("Ruta app_exports negocio:", APP_EXPORTS_PATH)


## 2. Funciones auxiliares

Estas funciones estandarizan el guardado de tablas y el formato de valores monetarios. De esta forma, las salidas quedan ordenadas en Parquet y CSV, lo que facilita su uso posterior en dashboards, reportes o aplicaciones.


In [0]:
def guardar_tabla_refined(df: pd.DataFrame, nombre_logico: str) -> None:
    """Guarda una tabla de negocio como Delta table en Unity Catalog y exporta CSV a app_exports."""
    sdf = spark.createDataFrame(df)
    save_spark_table(sdf, nombre_logico, export_csv=True)
    print(f"Tabla guardada: {nombre_logico}")
    print(f"  - dimensiones: {df.shape}")


def exportar_csv_app(df: pd.DataFrame, nombre_archivo: str) -> None:
    """Exporta una tabla en CSV para consumo de prototipos o dashboards."""
    APP_EXPORTS_PATH.mkdir(parents=True, exist_ok=True)
    salida = APP_EXPORTS_PATH / nombre_archivo
    df.to_csv(salida, index=False)
    print(f"Exportado para app: {salida}")


def millones_cop(valor):
    """Convierte pesos colombianos a millones de COP."""
    return valor / 1_000_000


def miles_cop(valor):
    """Convierte pesos colombianos a miles de COP."""
    return valor / 1_000


## 3. Validación de entradas disponibles

Antes de construir la capa de negocio se verifica que existan las tablas necesarias. La entrada principal es la matriz completa de modelado, porque permite representar toda la cohorte analítica y no solo el subconjunto de prueba.


In [0]:
entradas = {
    "Matriz completa de modelado": MATRIZ_MODELADO_TABLE,
    "Predicciones del mejor modelo en test": PREDICCIONES_TEST_TABLE,
}

for nombre, tabla in entradas.items():
    existe = table_exists_databricks(tabla)
    print(f"{nombre}: {existe} -> {full_table_name(tabla)}")

if not table_exists_databricks(MATRIZ_MODELADO_TABLE):
    raise FileNotFoundError(
        "No se encontró la matriz completa de modelado. "
        "Ejecute primero los notebooks 02, 03, 04 y 05."
    )


## 4. Carga de la cohorte analítica

Se toma la matriz final de modelado y se conserva una vista mínima con identificadores y tipo de cáncer. Esta será la base para construir el escenario de negocio por EPS, IPS, ruta diagnóstica y costos estimados.


In [0]:
pdf_matriz = load_spark_table(MATRIZ_MODELADO_TABLE).toPandas().copy()

print("Matriz completa cargada:", pdf_matriz.shape)
print("Columnas disponibles:", len(pdf_matriz.columns))

columnas_requeridas = ["sample_id", "patient_id", "cancer_type"]
faltantes = [col for col in columnas_requeridas if col not in pdf_matriz.columns]
if faltantes:
    raise ValueError(f"Faltan columnas requeridas en la matriz de modelado: {faltantes}")

pdf_cohorte = (
    pdf_matriz[columnas_requeridas]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Cohorte base:", pdf_cohorte.shape)
print("Pacientes únicos:", pdf_cohorte["patient_id"].nunique())
print("Muestras únicas:", pdf_cohorte["sample_id"].nunique())
print("Tipos de cáncer:", pdf_cohorte["cancer_type"].nunique())

display(pdf_cohorte.head())


## 5. Construcción del contexto operativo EPS/IPS

Esta sección agrega variables de negocio simuladas y coherentes con un escenario de red oncológica: EPS, IPS de referencia, departamento, régimen, edad, sexo, tiempos de ruta diagnóstica y costos estimados.

La lógica permite responder la pregunta del proyecto desde una perspectiva operativa: **¿cómo podría una solución analítica apoyar la optimización del diagnóstico oncológico en una red de atención?**


In [0]:
eps_colombia = [
    "Nueva EPS",
    "EPS Sanitas",
    "EPS SURA",
    "Salud Total EPS",
    "Famisanar EPS",
    "Compensar EPS",
    "Coosalud EPS",
    "Mutual Ser EPS",
]

ips_oncologicas = [
    "Instituto Nacional de Cancerología",
    "Clínica Las Américas Auna",
    "Clínica del Country",
    "Fundación Valle del Lili",
    "Clínica Imbanaco",
    "Hospital Pablo Tobón Uribe",
    "Clínica Foscal",
    "Centro Oncológico de Antioquia",
]

departamentos = [
    "Antioquia",
    "Bogotá D.C.",
    "Valle del Cauca",
    "Atlántico",
    "Santander",
    "Bolívar",
    "Cundinamarca",
    "Risaralda",
]

regimenes = ["Contributivo", "Subsidiado"]


def asignar_sexo(cancer_type: str) -> str:
    """Asigna sexo simulado con reglas básicas según el tipo de cáncer."""
    cancer_type = str(cancer_type).upper()

    if cancer_type in ["PRAD", "TGCT"]:
        return "M"
    if cancer_type in ["OV", "UCEC", "CESC"]:
        return "F"
    if cancer_type == "BRCA":
        return rng.choice(["F", "M"], p=[0.98, 0.02])
    return rng.choice(["F", "M"], p=[0.52, 0.48])


def asignar_edad(cancer_type: str) -> int:
    """Asigna edad simulada usando rangos plausibles por grupo tumoral."""
    cancer_type = str(cancer_type).upper()

    if cancer_type == "TGCT":
        edad = rng.normal(34, 8)
    elif cancer_type in ["THCA", "CESC"]:
        edad = rng.normal(47, 11)
    elif cancer_type in ["PRAD", "LUAD", "LUSC", "COAD", "READ", "STAD", "PAAD"]:
        edad = rng.normal(66, 10)
    else:
        edad = rng.normal(59, 12)

    return int(np.clip(round(edad), 18, 90))


def grupo_etario(edad: int) -> str:
    if edad < 40:
        return "18-39"
    if edad < 60:
        return "40-59"
    if edad < 75:
        return "60-74"
    return "75+"


pdf_negocio = pdf_cohorte.copy()
n = len(pdf_negocio)

pdf_negocio["edad"] = [asignar_edad(ct) for ct in pdf_negocio["cancer_type"]]
pdf_negocio["grupo_etario"] = pdf_negocio["edad"].apply(grupo_etario)
pdf_negocio["sexo"] = [asignar_sexo(ct) for ct in pdf_negocio["cancer_type"]]

pdf_negocio["departamento"] = rng.choice(
    departamentos,
    size=n,
    p=[0.24, 0.22, 0.16, 0.10, 0.10, 0.07, 0.07, 0.04],
)

pdf_negocio["regimen"] = rng.choice(regimenes, size=n, p=[0.58, 0.42])
pdf_negocio["eps"] = rng.choice(eps_colombia, size=n)
pdf_negocio["ips_referencia"] = rng.choice(ips_oncologicas, size=n)

print("Contexto operativo creado:", pdf_negocio.shape)
display(pdf_negocio.head())


## 6. Simulación de impacto en tiempos y costos

Se construyen indicadores estimados para comparar una ruta diagnóstica tradicional frente a una ruta apoyada por analítica.  
A diferencia de una simulación homogénea, aquí se asignan perfiles diferenciados por EPS para que el análisis muestre contrastes visibles: algunas EPS parten de rutas más largas y costosas, otras tienen mejor oportunidad inicial, y otras concentran mayor potencial de reducción con apoyo analítico.

El propósito no es afirmar un ahorro real, sino construir un escenario de valor para discusión con EPS e IPS.


In [0]:
# Perfiles operativos diferenciados por EPS.
# Estos valores son simulados y sirven para que el escenario de negocio tenga contraste visible.
# La idea es que cada EPS tenga un comportamiento distinto en oportunidad, costo y potencial de mejora.

parametros_eps = {
    "Nueva EPS": {
        "dias_base": 66, "sd_dias": 10,
        "reduccion_base": 17, "sd_reduccion": 4,
        "costo_base": 6_200_000, "sd_costo": 850_000,
        "ahorro_pct_base": 0.18, "sd_ahorro": 0.035,
    },
    "EPS Sanitas": {
        "dias_base": 49, "sd_dias": 8,
        "reduccion_base": 9, "sd_reduccion": 3,
        "costo_base": 4_250_000, "sd_costo": 550_000,
        "ahorro_pct_base": 0.11, "sd_ahorro": 0.025,
    },
    "EPS SURA": {
        "dias_base": 43, "sd_dias": 7,
        "reduccion_base": 7, "sd_reduccion": 2,
        "costo_base": 3_850_000, "sd_costo": 500_000,
        "ahorro_pct_base": 0.09, "sd_ahorro": 0.020,
    },
    "Salud Total EPS": {
        "dias_base": 58, "sd_dias": 9,
        "reduccion_base": 14, "sd_reduccion": 3,
        "costo_base": 5_250_000, "sd_costo": 750_000,
        "ahorro_pct_base": 0.15, "sd_ahorro": 0.030,
    },
    "Famisanar EPS": {
        "dias_base": 62, "sd_dias": 10,
        "reduccion_base": 19, "sd_reduccion": 4,
        "costo_base": 5_850_000, "sd_costo": 800_000,
        "ahorro_pct_base": 0.22, "sd_ahorro": 0.035,
    },
    "Compensar EPS": {
        "dias_base": 46, "sd_dias": 7,
        "reduccion_base": 8, "sd_reduccion": 2,
        "costo_base": 4_050_000, "sd_costo": 520_000,
        "ahorro_pct_base": 0.10, "sd_ahorro": 0.020,
    },
    "Coosalud EPS": {
        "dias_base": 74, "sd_dias": 12,
        "reduccion_base": 23, "sd_reduccion": 5,
        "costo_base": 7_100_000, "sd_costo": 950_000,
        "ahorro_pct_base": 0.25, "sd_ahorro": 0.040,
    },
    "Mutual Ser EPS": {
        "dias_base": 70, "sd_dias": 11,
        "reduccion_base": 21, "sd_reduccion": 5,
        "costo_base": 6_750_000, "sd_costo": 900_000,
        "ahorro_pct_base": 0.24, "sd_ahorro": 0.040,
    },
}

# Ajustes por complejidad tumoral. Estos modificadores ayudan a que también exista variación
# por tipo de cáncer, sin perder el efecto principal por EPS.
ajuste_cancer = {
    "GBM": {"dias": 7, "costo_factor": 1.20},
    "PAAD": {"dias": 6, "costo_factor": 1.18},
    "OV": {"dias": 5, "costo_factor": 1.15},
    "LUAD": {"dias": 4, "costo_factor": 1.12},
    "LUSC": {"dias": 4, "costo_factor": 1.12},
    "STAD": {"dias": 4, "costo_factor": 1.10},
    "COAD": {"dias": 3, "costo_factor": 1.08},
    "READ": {"dias": 3, "costo_factor": 1.08},
    "BRCA": {"dias": 1, "costo_factor": 1.00},
    "THCA": {"dias": -3, "costo_factor": 0.88},
    "PRAD": {"dias": -2, "costo_factor": 0.92},
}

# Ajustes por régimen: en el escenario simulado, el régimen subsidiado tiene mayor fricción operativa.
ajuste_regimen_dias = {"Contributivo": 0, "Subsidiado": 6}
ajuste_regimen_costo = {"Contributivo": 1.00, "Subsidiado": 1.08}

dias_actuales = []
dias_con_analitica = []
costos_actuales = []
costos_con_analitica = []
porcentajes_ahorro = []

for _, row in pdf_negocio.iterrows():
    eps = row["eps"]
    cancer_type = str(row["cancer_type"]).upper()
    regimen = row["regimen"]

    p = parametros_eps[eps]
    ajuste_ct = ajuste_cancer.get(cancer_type, {"dias": 0, "costo_factor": 1.00})

    # Ruta actual con efecto por EPS, tipo de cáncer y régimen.
    dias_actual = rng.normal(p["dias_base"], p["sd_dias"])
    dias_actual += ajuste_ct["dias"]
    dias_actual += ajuste_regimen_dias[regimen]
    dias_actual = int(np.clip(round(dias_actual), 18, 115))

    # Reducción con analítica: también depende de la EPS.
    reduccion = rng.normal(p["reduccion_base"], p["sd_reduccion"])
    # En rutas más largas, el potencial de mejora aumenta ligeramente.
    if dias_actual > 70:
        reduccion += 4
    elif dias_actual < 45:
        reduccion -= 2
    reduccion = int(np.clip(round(reduccion), 4, 34))

    dias_analitica = int(max(8, dias_actual - reduccion))

    # Costo actual con efecto por EPS, cáncer y régimen.
    costo_actual = rng.normal(p["costo_base"], p["sd_costo"])
    costo_actual *= ajuste_ct["costo_factor"]
    costo_actual *= ajuste_regimen_costo[regimen]
    costo_actual = int(np.clip(round(costo_actual, -3), 2_200_000, 10_500_000))

    # Porcentaje de ahorro: diferenciado por EPS y limitado a rangos plausibles.
    ahorro_pct = rng.normal(p["ahorro_pct_base"], p["sd_ahorro"])
    if dias_actual > 70:
        ahorro_pct += 0.03
    ahorro_pct = float(np.clip(ahorro_pct, 0.06, 0.33))

    costo_analitica = int(round(costo_actual * (1 - ahorro_pct), -3))

    dias_actuales.append(dias_actual)
    dias_con_analitica.append(dias_analitica)
    costos_actuales.append(costo_actual)
    costos_con_analitica.append(costo_analitica)
    porcentajes_ahorro.append(ahorro_pct)

pdf_negocio["dias_ruta_diagnostica_actual"] = dias_actuales
pdf_negocio["dias_ruta_diagnostica_con_analitica"] = dias_con_analitica
pdf_negocio["dias_reducidos"] = (
    pdf_negocio["dias_ruta_diagnostica_actual"]
    - pdf_negocio["dias_ruta_diagnostica_con_analitica"]
)

pdf_negocio["costo_ruta_actual_cop"] = costos_actuales
pdf_negocio["costo_ruta_con_analitica_cop"] = costos_con_analitica
pdf_negocio["ahorro_estimado_cop"] = (
    pdf_negocio["costo_ruta_actual_cop"]
    - pdf_negocio["costo_ruta_con_analitica_cop"]
)
pdf_negocio["porcentaje_ahorro_estimado"] = porcentajes_ahorro

# Indicadores operativos para priorización.
pdf_negocio["ruta_actual_mayor_62_dias"] = pdf_negocio["dias_ruta_diagnostica_actual"] > 62
pdf_negocio["prioridad_operativa"] = np.select(
    [
        pdf_negocio["ruta_actual_mayor_62_dias"] & (pdf_negocio["dias_reducidos"] >= 18),
        pdf_negocio["ruta_actual_mayor_62_dias"] | (pdf_negocio["dias_reducidos"] >= 12),
    ],
    ["Alta", "Media"],
    default="Baja",
)

print("Variables de impacto creadas:", pdf_negocio.shape)

# Vista rápida para verificar que ya no quedan valores casi iguales por EPS.
verificacion_eps = (
    pdf_negocio
    .groupby("eps", as_index=False)
    .agg(
        dias_actual=("dias_ruta_diagnostica_actual", "mean"),
        dias_con_analitica=("dias_ruta_diagnostica_con_analitica", "mean"),
        reduccion_dias=("dias_reducidos", "mean"),
        costo_actual_millones=("costo_ruta_actual_cop", lambda x: x.mean() / 1_000_000),
        costo_con_analitica_millones=("costo_ruta_con_analitica_cop", lambda x: x.mean() / 1_000_000),
        ahorro_promedio_miles=("ahorro_estimado_cop", lambda x: x.mean() / 1_000),
        pct_mayor_62=("ruta_actual_mayor_62_dias", lambda x: x.mean() * 100),
    )
    .round(2)
    .sort_values("reduccion_dias", ascending=False)
)

display(verificacion_eps)
display(pdf_negocio.head())


## 7. Guardado de la tabla de negocio

La tabla `refined_contexto_negocio_eps_ips_cohorte` queda como insumo principal para análisis de negocio, visualización y construcción del relato de impacto.


In [0]:
guardar_tabla_refined(
    pdf_negocio,
    CONTEXTO_NEGOCIO_TABLE,
)

exportar_csv_app(pdf_negocio, "contexto_negocio_eps_ips_cohorte.csv")


## 8. Resumen ejecutivo general

Este resumen consolida las cifras principales para la exposición y el informe: tamaño de la cohorte, cobertura simulada de EPS/IPS, tipos de cáncer, tiempos promedio y ahorro estimado.


In [0]:
resumen_cohorte = pd.DataFrame([
    {
        "total_muestras_analizadas": len(pdf_negocio),
        "pacientes_unicos": pdf_negocio["patient_id"].nunique(),
        "eps_analizadas": pdf_negocio["eps"].nunique(),
        "ips_referencia": pdf_negocio["ips_referencia"].nunique(),
        "tipos_cancer": pdf_negocio["cancer_type"].nunique(),
        "dias_promedio_ruta_actual": round(pdf_negocio["dias_ruta_diagnostica_actual"].mean(), 2),
        "dias_promedio_ruta_con_analitica": round(pdf_negocio["dias_ruta_diagnostica_con_analitica"].mean(), 2),
        "reduccion_promedio_dias": round(pdf_negocio["dias_reducidos"].mean(), 2),
        "pct_rutas_actuales_mayor_62_dias": round(pdf_negocio["ruta_actual_mayor_62_dias"].mean() * 100, 2),
        "costo_promedio_actual_cop": round(pdf_negocio["costo_ruta_actual_cop"].mean(), 0),
        "costo_promedio_con_analitica_cop": round(pdf_negocio["costo_ruta_con_analitica_cop"].mean(), 0),
        "ahorro_promedio_estimado_cop": round(pdf_negocio["ahorro_estimado_cop"].mean(), 0),
        "ahorro_total_estimado_cop": round(pdf_negocio["ahorro_estimado_cop"].sum(), 0),
    }
])

display(resumen_cohorte)

guardar_tabla_refined(
    resumen_cohorte,
    RESUMEN_EJECUTIVO_TABLE,
)

exportar_csv_app(resumen_cohorte, "resumen_ejecutivo_eps_ips_cohorte.csv")


## 9. Resumen por EPS

Este resumen permite identificar diferencias operativas entre aseguradoras simuladas: volumen de casos, variedad de tipos de cáncer, reducción promedio de días y ahorro estimado.


In [0]:
resumen_eps = (
    pdf_negocio
    .groupby("eps", as_index=False)
    .agg(
        muestras=("sample_id", "count"),
        pacientes_unicos=("patient_id", "nunique"),
        tipos_cancer=("cancer_type", "nunique"),
        ips_referencia=("ips_referencia", "nunique"),
        dias_promedio_actual=("dias_ruta_diagnostica_actual", "mean"),
        dias_promedio_con_analitica=("dias_ruta_diagnostica_con_analitica", "mean"),
        reduccion_promedio_dias=("dias_reducidos", "mean"),
        pct_rutas_actuales_mayor_62_dias=("ruta_actual_mayor_62_dias", lambda x: x.mean() * 100),
        costo_promedio_actual_cop=("costo_ruta_actual_cop", "mean"),
        costo_promedio_con_analitica_cop=("costo_ruta_con_analitica_cop", "mean"),
        ahorro_promedio_estimado_cop=("ahorro_estimado_cop", "mean"),
        ahorro_total_estimado_cop=("ahorro_estimado_cop", "sum"),
    )
)

cols_num = resumen_eps.select_dtypes(include="number").columns
resumen_eps[cols_num] = resumen_eps[cols_num].round(2)
resumen_eps = resumen_eps.sort_values("ahorro_total_estimado_cop", ascending=False).reset_index(drop=True)

display(resumen_eps)

guardar_tabla_refined(
    resumen_eps,
    RESUMEN_EPS_TABLE,
)

exportar_csv_app(resumen_eps, "resumen_por_eps_cohorte.csv")


## 10. Resumen por tipo de cáncer

Este resumen conecta la dimensión molecular con el valor operativo: permite observar qué tipos de cáncer concentran más muestras, mayor reducción esperada de tiempos o mayor ahorro promedio estimado por caso.


In [0]:
resumen_cancer = (
    pdf_negocio
    .groupby("cancer_type", as_index=False)
    .agg(
        muestras=("sample_id", "count"),
        pacientes_unicos=("patient_id", "nunique"),
        edad_promedio=("edad", "mean"),
        eps=("eps", "nunique"),
        ips_referencia=("ips_referencia", "nunique"),
        dias_promedio_actual=("dias_ruta_diagnostica_actual", "mean"),
        dias_promedio_con_analitica=("dias_ruta_diagnostica_con_analitica", "mean"),
        reduccion_promedio_dias=("dias_reducidos", "mean"),
        pct_rutas_actuales_mayor_62_dias=("ruta_actual_mayor_62_dias", lambda x: x.mean() * 100),
        costo_promedio_actual_cop=("costo_ruta_actual_cop", "mean"),
        costo_promedio_con_analitica_cop=("costo_ruta_con_analitica_cop", "mean"),
        ahorro_promedio_estimado_cop=("ahorro_estimado_cop", "mean"),
        ahorro_total_estimado_cop=("ahorro_estimado_cop", "sum"),
    )
)

cols_num = resumen_cancer.select_dtypes(include="number").columns
resumen_cancer[cols_num] = resumen_cancer[cols_num].round(2)
resumen_cancer = resumen_cancer.sort_values("muestras", ascending=False).reset_index(drop=True)

display(resumen_cancer)

guardar_tabla_refined(
    resumen_cancer,
    RESUMEN_CANCER_TABLE,
)

exportar_csv_app(resumen_cancer, "resumen_por_tipo_cancer_cohorte.csv")


## 11. Visualizaciones para comunicación de negocio

Las siguientes visualizaciones están pensadas para soportar el informe y la exposición.  
En esta versión se resaltan las EPS con mejor desempeño relativo y se usan datos más dispersos para que las diferencias operativas sean visibles: mayor ahorro, mayor reducción de días, mayor costo inicial y brechas entre ruta actual y ruta con apoyo analítico.


In [0]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#334155",
    "axes.labelcolor": "#1f2937",
    "xtick.color": "#1f2937",
    "ytick.color": "#1f2937",
    "text.color": "#111827",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
})


def guardar_figura(nombre: str) -> None:
    ruta = VIZ_PATH / nombre
    plt.tight_layout()
    plt.savefig(ruta, dpi=300, bbox_inches="tight")
    print("Figura guardada:", ruta.resolve())
    plt.show()


In [0]:
# 1. Ahorro promedio estimado por EPS
# Se resalta la EPS con mayor ahorro promedio por caso.

df_plot = resumen_eps.sort_values("ahorro_promedio_estimado_cop", ascending=True).copy()
df_plot["ahorro_promedio_miles"] = miles_cop(df_plot["ahorro_promedio_estimado_cop"])

mejor_eps = df_plot.loc[df_plot["ahorro_promedio_estimado_cop"].idxmax(), "eps"]
colores = ["#f59e0b" if eps == mejor_eps else "#2563eb" for eps in df_plot["eps"]]

plt.figure(figsize=(10, 6))
plt.barh(df_plot["eps"], df_plot["ahorro_promedio_miles"], color=colores)
plt.xlabel("Ahorro promedio estimado por caso (miles COP)")
plt.ylabel("EPS")
plt.title("Ahorro promedio estimado por caso según EPS")

for i, valor in enumerate(df_plot["ahorro_promedio_miles"]):
    plt.text(valor + 15, i, f"{valor:,.0f}", va="center", fontsize=9)

plt.xlim(0, df_plot["ahorro_promedio_miles"].max() * 1.18)
guardar_figura("01_ahorro_promedio_por_eps.png")


In [0]:
# 2. Reducción promedio de días por EPS
# Se resalta la EPS con mayor reducción estimada de la ruta diagnóstica.

df_plot = resumen_eps.sort_values("reduccion_promedio_dias", ascending=True).copy()

mejor_eps = df_plot.loc[df_plot["reduccion_promedio_dias"].idxmax(), "eps"]
colores = ["#16a34a" if eps == mejor_eps else "#2563eb" for eps in df_plot["eps"]]

plt.figure(figsize=(10, 6))
plt.barh(df_plot["eps"], df_plot["reduccion_promedio_dias"], color=colores)
plt.xlabel("Reducción promedio de días")
plt.ylabel("EPS")
plt.title("Reducción promedio estimada de la ruta diagnóstica por EPS")

for i, valor in enumerate(df_plot["reduccion_promedio_dias"]):
    plt.text(valor + 0.3, i, f"{valor:.1f}", va="center", fontsize=9)

plt.xlim(0, df_plot["reduccion_promedio_dias"].max() * 1.18)
guardar_figura("02_reduccion_promedio_dias_por_eps.png")


In [0]:
# 3. Comparación de costo promedio: ruta actual vs ruta con analítica
# Se ordena por ahorro promedio para que la brecha entre rutas sea más visible.

df_costos = resumen_eps.sort_values("ahorro_promedio_estimado_cop", ascending=False).copy()
x = np.arange(len(df_costos))
width = 0.38

costo_actual_m = millones_cop(df_costos["costo_promedio_actual_cop"])
costo_analitica_m = millones_cop(df_costos["costo_promedio_con_analitica_cop"])
ahorro_m = costo_actual_m - costo_analitica_m

plt.figure(figsize=(12, 6))
plt.bar(
    x - width / 2,
    costo_actual_m,
    width,
    label="Ruta actual",
    color="#2563eb",
)
plt.bar(
    x + width / 2,
    costo_analitica_m,
    width,
    label="Con apoyo analítico",
    color="#f97316",
)

for i, ahorro in enumerate(ahorro_m):
    y = max(costo_actual_m.iloc[i], costo_analitica_m.iloc[i]) + 0.12
    plt.text(i, y, f"Ahorro: {ahorro:.1f}M", ha="center", fontsize=8)

plt.xticks(x, df_costos["eps"], rotation=45, ha="right")
plt.ylabel("Costo promedio por caso (millones COP)")
plt.title("Costo promedio estimado de ruta diagnóstica por EPS")
plt.legend()
plt.ylim(0, costo_actual_m.max() * 1.25)
guardar_figura("03_comparacion_costos_promedio_por_eps.png")


In [0]:
# 4. Distribución de muestras por tipo de cáncer

df_plot = resumen_cancer.sort_values("muestras", ascending=True).copy()

plt.figure(figsize=(10, 7))
plt.barh(df_plot["cancer_type"], df_plot["muestras"])
plt.xlabel("Número de muestras")
plt.ylabel("Tipo de cáncer")
plt.title("Distribución de muestras por tipo de cáncer")
guardar_figura("04_muestras_por_tipo_cancer.png")


In [0]:
# 5. Ahorro promedio estimado por tipo de cáncer

df_plot = resumen_cancer.sort_values("ahorro_promedio_estimado_cop", ascending=True).copy()
df_plot["ahorro_promedio_miles"] = miles_cop(df_plot["ahorro_promedio_estimado_cop"])

plt.figure(figsize=(10, 7))
plt.barh(df_plot["cancer_type"], df_plot["ahorro_promedio_miles"])
plt.xlabel("Ahorro promedio estimado por caso (miles COP)")
plt.ylabel("Tipo de cáncer")
plt.title("Ahorro promedio estimado por tipo de cáncer")
guardar_figura("05_ahorro_promedio_por_tipo_cancer.png")


In [0]:
# 6. Mapa de distribución EPS vs tipo de cáncer

tabla_eps_cancer = pd.crosstab(pdf_negocio["eps"], pdf_negocio["cancer_type"])

plt.figure(figsize=(13, 6))
plt.imshow(tabla_eps_cancer, aspect="auto")
plt.colorbar(label="Número de muestras")
plt.xticks(
    ticks=np.arange(len(tabla_eps_cancer.columns)),
    labels=tabla_eps_cancer.columns,
    rotation=45,
    ha="right",
)
plt.yticks(
    ticks=np.arange(len(tabla_eps_cancer.index)),
    labels=tabla_eps_cancer.index,
)
plt.xlabel("Tipo de cáncer")
plt.ylabel("EPS")
plt.title("Distribución de muestras por EPS y tipo de cáncer")
guardar_figura("06_mapa_eps_tipo_cancer.png")


## 12. Validación final de salidas

Se listan las tablas y visualizaciones generadas para verificar que el componente de negocio quedó completo y reproducible.


In [0]:
tablas_generadas = [
    CONTEXTO_NEGOCIO_TABLE,
    RESUMEN_EJECUTIVO_TABLE,
    RESUMEN_EPS_TABLE,
    RESUMEN_CANCER_TABLE,
]

print("Tablas refined generadas en Unity Catalog:")
for tabla in tablas_generadas:
    print("-", tabla, "->", table_exists_databricks(tabla), full_table_name(tabla))

print("\nVisualizaciones generadas:")
for ruta in sorted(VIZ_PATH.glob("*.png")):
    print("-", ruta.name)

print("\nExports para app:")
for ruta in sorted(APP_EXPORTS_PATH.glob("*.csv")):
    print("-", ruta.name)
